# 1. Set up Graphistry Credentials

In [76]:
from dotenv import load_dotenv
import os

# Load environment variables from ".env" file
load_dotenv("env")

myusername = os.getenv("GRAPHISTRY_USERNAME")
mypassword = os.getenv("GRAPHISTRY_PASSWORD")

# 2. Register with PyGraphistry

In [77]:
import graphistry

graphistry.register(api=3, server='hub.graphistry.com', username=myusername, password=mypassword)


# 3. Loading Data

In [79]:
import pandas as pd
import graphistry

# Load the dataset using Pandas
df_products = pd.read_parquet('../data/refined_knowledge_graph/products.parquet')
df_products.head(10).to_csv('10products.csv', index=False)

df_edges = pd.read_parquet('../data/refined_knowledge_graph/edges.parquet')
df_edges.head(10).to_csv('10edges.csv', index=False)

## A. Using Top10 rows for easier visualization

In [80]:
# Load top 10 rows (products)
df_top10products = pd.read_csv('10products.csv')

# Add a dummy column for visualization purposes
df_top10products["buyer_org"] = "US Gov"

# Print the first 10 rows
print(df_top10products[['name', 'company_name', 'buyer_org']])


               name company_name buyer_org
0           ChatGPT       OpenAI    US Gov
1            Claude    Anthropic    US Gov
2  Claude AI models    Anthropic    US Gov
3     Claude Sonnet    Anthropic    US Gov
4            Cursor    Anysphere    US Gov
5            Dall-E       OpenAI    US Gov
6            GPT-4o       OpenAI    US Gov
7   Google Shopping       Google    US Gov
8              Haul       Amazon    US Gov
9         Pinterest    Pinterest    US Gov


In [81]:
# Load top 10 rows (edges)
df_top10edges = pd.read_csv('10edges.csv')

# Print the first 10 rows
print(df_top10edges)

          src                 dst relationship
0      Amazon                Haul        Sells
1  Apple Inc.    ultrathin iPhone        Sells
2  Apple Inc.      foldable phone        Sells
3  Apple Inc.       iPhone 17 Pro        Sells
4  Apple Inc.  standard iPhone 17        Sells
5   Microsoft                Xbox        Sells
6  Apple Inc.              iPhone        Sells
7  Apple Inc.   iPhone 17 Pro Max        Sells
8   Anthropic       Claude Sonnet        Sells
9  Apple Inc.          iPhone 16e        Sells


# 4. Plot

## A. Simple graph

In [82]:
g = graphistry.edges(df_top10products, 'company_name', 'buyer_org')
g.plot() # Make sure you called graphistry.register() above

## B. Hypergraphs

### I. Approach 1: Treat each row as a node, and link it to each cell value in it

In [83]:
hg1 = graphistry.hypergraph(
    df_top10edges,

    # Optional: Subset of columns to turn into nodes; defaults to all
    entity_types=['src', 'dst'],
)

hg1_g = hg1['graph']
hg1_g.plot()

# links 20
# events 10
# attrib entities 14


### II. Approach 2: Link values from column entries

In [84]:
hg2 = graphistry.hypergraph(
    df_top10edges,
    entity_types=['src', 'dst', 'relationship'],
    direct=True,
    opts={
        # Optional: Without, creates edges that are all-to-all for each row
        'EDGES': {
            'src' : ['dst', 'relationship'],
            'relationship' : ['dst'],
        },
    }
)

hg2_g = hg2['graph']
hg2_g.plot()

# links 30
# events 10
# attrib entities 15


## C. Advanced Plotting

*You can then drive visual style based on node and edge attribute*

In [85]:
# Cell:
# Compute nodes_df by combining entities in company_name and buyer_org
# As part of this, compute product counts for each node

buyers_df = (
    df_top10products[['buyer_org']]
    .drop_duplicates()
    .rename(columns={'buyer_org': 'node_id'})
    .assign(type='buyer')
)


sellers_df = (
    df_top10products
    .groupby(['company_name'])
    .agg(products=pd.NamedAgg(column='company_name', aggfunc='count'))
    .reset_index()
    .rename(columns={'company_name': 'node_id'}).assign(type='seller')
)

nodes_df = pd.concat([buyers_df, sellers_df])
nodes_df.sort_values(by='products', ascending=False)

,node_id,type,products
1,Anthropic,seller,3.0
4,OpenAI,seller,3.0
0,Amazon,seller,1.0
2,Anysphere,seller,1.0
3,Google,seller,1.0
5,Pinterest,seller,1.0
0,US Gov,buyer,NaN


In [86]:
# Cell:
# Add

# New encodings features requires api=3: `graphistry.register(api=3, username='...', password='...')

g2 = (g
      .nodes(nodes_df, 'node_id')

      # 'red', '#f00', '#ff0000'
      .encode_point_color('type', categorical_mapping={
          'seller': 'green',
          'buyer': 'red',
        }, default_mapping='gray')

      # Icons: https://fontawesome.com/v4.7/cheatsheet/
      .encode_point_icon('type', categorical_mapping={
          'seller': 'industry',
          'buyer': 'handshake'
        })

      .encode_point_size('products')

      .addStyle(bg={'color': '#eee'}, page={'title': 'My Graph'})

      # Options: https://hub.graphistry.com/docs/api/1/rest/url/
      .settings(url_params={'play': 1000, 'pointSize': 0.5})
    )

g2.plot(as_files=False)

### I. Advanced bindings with Hypergraphs

In [87]:
hg2_g._nodes.sample(3)

,src,nodeTitle,type,category,nodeID,dst,relationship,EventID
14,NaN,Sells,relationship,relationship,relationship::Sells,NaN,Sells,NaN
5,NaN,ultrathin iPhone,dst,dst,dst::ultrathin iPhone,ultrathin iPhone,NaN,NaN
4,NaN,Haul,dst,dst,dst::Haul,Haul,NaN,NaN


In [88]:
hg2_g._edges.sample(3)

,src,dst,EventID,relationship,edgeType
4,relationship::Sells,dst::standard iPhone 17,EventID::4,Sells,relationship::dst
28,src::Anthropic,relationship::Sells,EventID::8,Sells,src::relationship
7,relationship::Sells,dst::iPhone 17 Pro Max,EventID::7,Sells,relationship::dst


In [89]:
(hg2_g

 .encode_point_color('type', categorical_mapping={
     'src': 'yellow',
     'dst': 'blue'
 }, default_mapping='gray')

 .encode_point_icon('type', categorical_mapping={
      'src': 'store',
      'dst': 'box-open'
 }, default_mapping='')

 .settings(url_params={'pointsOfInterestMax': 10})

).plot()